In [4]:
from perceval import RemoteProcessor, Processor
from perceval.algorithm import Sampler
import perceval as pcvl

from photon_catalysis.state_preparation_circuit import StatePreparationCircuit
from photon_catalysis.optimal_preparation import optimal_preparation
from photon_catalysis.utils import kets_to_state_dict

token = 'your token here'

In [5]:
# make photon catalysis state
state = kets_to_state_dict([(2, 0, 0), (0, 2, 0), (0, 0, 2)])

# find the preparation recipy with 1 catalysis photon and highest probability of success
# the theoretical fidelity (assuming perfect photon additions) is 1
W, _, _ = max(
    optimal_preparation(
        state,
        extra_photons=1,
        num_decompositions=25,
    ),
    key=lambda t: abs(t[1]),
)

Optimizing probability...: 100%|██████████| 10000/10000 [00:00<00:00, 26379.22it/s, prob=0.29081693, scale=(-0.12533335+0.7545925j)]


In [21]:
# construct abstract preparation circuit
circuit = StatePreparationCircuit(W, state)

# construct parceval circuit, input state, and postselection condition
circuit, input_state, post_select = circuit.to_perceval(
    photon_addition_r=0.6
)

In [22]:
# number of shots for the sampler
num_shots = 1000000

In [23]:
# set up simulation for Ascella machine
sim_processor = RemoteProcessor('sim:ascella', token)
sim_processor.set_circuit(circuit)
sim_processor.min_detected_photons_filter(2)
sim_processor.with_input(input_state)
sim_processor.set_postselection(post_select)
sim_sampler = Sampler(sim_processor, max_shots_per_call=num_shots)
sim_samples = sim_sampler.sample_count(num_shots)['results']

In [24]:
print(f"Samples: {sim_samples}")

Samples: {
  |0,0,0,1,1,0,0>: 734
  |0,0,0,1,0,1,0>: 1090
  |0,0,0,1,0,0,1>: 1121
  |0,0,0,1,1,0,1>: 1
  |0,0,0,1,1,1,0>: 1
}


In [25]:
# run on the actual QPU
qpu_processor = RemoteProcessor('qpu:ascella', token)
qpu_processor.set_circuit(circuit)
qpu_processor.min_detected_photons_filter(2)
qpu_processor.with_input(input_state)
qpu_processor.set_postselection(post_select)
qpu_sampler = Sampler(qpu_processor, max_shots_per_call=num_shots)
qpu_samples = qpu_sampler.sample_count(num_shots)['results']

In [26]:
print(f"Samples: {qpu_samples}")

Samples: {
  |0,0,0,1,0,1,0>: 416
  |0,0,0,1,1,0,0>: 330
  |0,0,0,1,0,0,1>: 467
}
